# `ProviderToolSearchMiddleware`

Middleware that defers selected agent tools behind a model provider's native tool-search capability.

Instead of sending the complete schema for every bound tool on every model call, the middleware marks selected tools with `extras["defer_loading"] = True` and appends the provider's server-side tool-search descriptor. The provider can then retrieve a deferred tool's full schema only when the model needs it.

This can reduce request size when an agent has many tools.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

> **Warning**
>
> Provider-native tool search is used only when at least one tool is deferred.  
> At this pinned source revision, only Anthropic and OpenAI are supported.
>
> If a tool is deferred and the provider cannot be identified—or the provider
> does not support server-side tool search—the model call raises `ValueError`.

## Type Alias

### `ToolIdentifier`

Represents a tool that should be deferred.

```python
ToolIdentifier: TypeAlias = str | BaseTool
```

A searchable tool may be supplied as:

* Its registered tool name.
* Its `BaseTool` instance.

## Provider Tool Descriptors

The module stores provider-native search descriptors in `_SERVER_TOOL_SEARCH_TOOLS`.

```python
_SERVER_TOOL_SEARCH_TOOLS = {
    "anthropic": {
        "type": "tool_search_tool_bm25_20251119",
        "name": "tool_search_tool_bm25",
    },
    "openai": {
        "type": "tool_search",
    },
}
```

These dictionaries are appended to the request's tool list when provider-side search is activated.

The provider identifiers are version-sensitive and may need updating when provider APIs change.

## Constructor

```python
ProviderToolSearchMiddleware(
    *,
    searchable_tools: list[ToolIdentifier] | None = None
)
```

## Parameters

* `searchable_tools` — Tools or tool names to defer behind provider-native search.
  * Default: `None`
  * Strings are interpreted as bound tool names.
  * `BaseTool` values are converted to their `.name`.
  * Duplicate names are removed because the values are stored in a set.

A tool can also be deferred without being listed here by defining:

```python
tool.extras = {
    "defer_loading": True
}
```

## Attributes

* `searchable_tool_names` — Set containing the names explicitly configured for deferral.
  * Type: `set[str]`
  * An empty set is stored when `searchable_tools=None`.

## Methods

1. `_prepare_request`: Validates searchable tools and prepares a request for provider-native tool search.
   * Confirms that every explicitly configured tool name belongs to a bound `BaseTool`.
   * Returns the original request unchanged when no bound tool is deferred.
   * Determines the model provider only when at least one tool is deferred.
   * Raises `ValueError` when the provider is unknown or unsupported.
   * Copies deferred tools with `extras["defer_loading"] = True`.
   * Appends the provider's native tool-search descriptor.
   - **Syntax:**
     ```python
     _prepare_request(
         self,
         request: ModelRequest[ContextT] # Request being prepared
     ) -> ModelRequest[ContextT]
     ```

2. `wrap_model_call`: Prepares and executes a synchronous model request.
   * Calls `_prepare_request`.
   * Passes the resulting request to the model-call handler.
   * Returns the handler's result.
   - **Syntax:**
     ```python
     wrap_model_call(
         self,
         request: ModelRequest[ContextT], # Original model request
         handler: Callable[
             [ModelRequest[ContextT]],
             ModelResponse[ResponseT]
         ] # Function that executes the model call
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

3. `awrap_model_call`: Asynchronous version of `wrap_model_call`.
   * Applies the same validation and request transformation.
   * Awaits the asynchronous model-call handler.
   - **Syntax:**
     ```python
     async def awrap_model_call(
         self,
         request: ModelRequest[ContextT], # Original model request
         handler: Callable[
             [ModelRequest[ContextT]],
             Awaitable[ModelResponse[ResponseT]]
         ] # Async function that executes the model call
     ) -> ModelResponse[ResponseT] | AIMessage
     ```

## Tool Deferral Rules

A tool is considered deferred when either condition is true:

```python
tool.extras["defer_loading"] is True
```

or:

```python
tool.name in searchable_tool_names
```

Only `BaseTool` instances can be deferred by this middleware.

Dictionary-based tool specifications—such as provider-native tools already present in the request—are left unchanged because they do not provide the `BaseTool` name and `extras` interface used here.

## Request Preparation Flow

```text
Receive ModelRequest
        |
        v
Validate explicitly named searchable tools
        |
        v
Is any BaseTool deferred?
        |
   No --+--> Return the original request unchanged
        |
       Yes
        |
        v
Infer the model provider
        |
        v
Is the provider Anthropic or OpenAI?
        |
   No --+--> Raise ValueError
        |
       Yes
        |
        v
Copy selected tools with defer_loading=True
        |
        v
Append provider-native tool-search descriptor
        |
        v
Execute the model call
```

## Explicit Name Validation

When `searchable_tools` contains names, every name must match a bound `BaseTool`.

For example:

```python
ProviderToolSearchMiddleware(
    searchable_tools=["lookup_order"]
)
```

requires a bound tool whose name is exactly:

```text
lookup_order
```

Otherwise, the middleware raises an error similar to:

```text
ProviderToolSearchMiddleware: searchable_tools references tool(s)
not bound to the model: lookup_order
```

Multiple unknown names are sorted and displayed as a comma-separated list.

The validation applies only to names explicitly supplied through `searchable_tools`. A bound tool that already has `defer_loading=True` does not require an entry in that list.

## Pass-Through Behaviour

When no tool is deferred, `_prepare_request` returns the same request object without:

* Attempting provider inference.
* Checking provider support.
* Adding a provider-native search tool.
* Copying or modifying bound tools.

Therefore, the middleware can be present with an unsupported provider as long as no tool is marked for deferred loading.

## Provider Support

At this pinned revision, the supported normalized providers are:

| Provider | Injected tool descriptor |
|---|---|
| Anthropic | `{"type": "tool_search_tool_bm25_20251119", "name": "tool_search_tool_bm25"}` |
| OpenAI | `{"type": "tool_search"}` |

Provider names are normalized by:

1. Converting the identifier to lowercase.
2. Replacing hyphens with underscores.

Example:

```text
"OPEN-AI" -> "open_ai"
```

Normalization does not automatically make an unsupported alias valid; the resulting value must still exist in `_SERVER_TOOL_SEARCH_TOOLS`.

## Provider Inference

The middleware attempts to infer the provider in this order:

1. Dynamic model parameters returned by the model's `_model_params(runtime.config)`.
2. The model's `_default_config`.
3. LangSmith parameters returned by `_get_ls_params()`, using `ls_provider`.
4. Known model class names.

When dynamic model parameters and default configuration are both dictionaries, they are merged before provider inference, with dynamic values taking precedence.

A malformed runtime configuration that is not a mapping is treated as `None` before `_model_params` is called.

### Parameter-Based Inference

The helper first checks:

```python
params["model_provider"]
```

When that is absent, it checks:

```python
params["model"]
```

A model string containing a provider prefix is split at the first colon:

```text
anthropic:claude-opus-4-8 -> anthropic
openai:gpt-5.5            -> openai
```

For an unprefixed model name, LangChain's internal `_attempt_infer_model_provider` helper is used.

Deprecation warnings produced during this internal lookup are suppressed because the result is used only for provider routing.

### Class-Name Fallback

The following class names are recognized directly:

```python
{
    "ChatAnthropic": "anthropic",
    "AnthropicChat": "anthropic",
    "ChatOpenAI": "openai",
    "OpenAIChat": "openai",
}
```

Other class names return `None`.

## Errors

### Searchable Tool Is Not Bound

Raised when an explicitly listed name does not match a bound `BaseTool`.

```text
ProviderToolSearchMiddleware: searchable_tools references tool(s)
not bound to the model: <tool names>
```

### Provider Cannot Be Determined

Raised when at least one tool is deferred but all provider-inference methods fail.

```text
ProviderToolSearchMiddleware could not determine the provider for model
'<model class>'; server-side tool search supports: anthropic, openai
```

### Provider Is Unsupported

Raised when a provider is identified but is absent from the supported-provider mapping.

```text
ProviderToolSearchMiddleware requires a provider with server-side tool
search, but got '<provider>'; supported providers: anthropic, openai
```

## Internal Types

### `_ServerToolSearchSpec`

Typed dictionary describing a provider-native tool-search tool.

```python
class _ServerToolSearchSpec(TypedDict):
    type: str
    name: NotRequired[str]
```

* `type` — Provider-specific search-tool type.
* `name` — Optional provider-specific tool name.

## Internal Helper Functions

### `_to_tool_names`

Converts tool names and `BaseTool` instances into a set of names.

```python
_to_tool_names(
    tools: list[ToolIdentifier] | None
) -> set[str]
```

Returns an empty set when `tools=None`.

### `_is_deferred_tool`

Checks whether a bound tool should be deferred.

```python
_is_deferred_tool(
    tool: BaseTool | dict[str, Any],
    tool_names: set[str]
) -> bool
```

Returns `False` for dictionary-based tool specifications.

### `_defer_tool_if_needed`

Returns a copied tool with deferred loading enabled when required.

```python
_defer_tool_if_needed(
    tool: BaseTool | dict[str, Any],
    tool_names: set[str]
) -> BaseTool | dict[str, Any]
```

For a deferred `BaseTool`, it merges the existing extras with:

```python
{
    "defer_loading": True
}
```

The original tool is not mutated; `model_copy` is used.

### `_get_model_provider`

Infers the normalized provider from the model and runtime.

```python
_get_model_provider(
    model: BaseChatModel,
    runtime: Any
) -> str | None
```

Returns `None` when the provider cannot be identified.

### `_provider_from_params`

Infers a provider from a parameter dictionary.

```python
_provider_from_params(
    params: dict[str, Any]
) -> str | None
```

It checks `model_provider` before `model`.

### `_provider_from_model_name`

Infers a provider from a prefixed or recognizable model name.

```python
_provider_from_model_name(
    model_name: str
) -> str | None
```

### `_provider_from_class_name`

Maps recognized chat-model class names to providers.

```python
_provider_from_class_name(
    class_name: str
) -> str | None
```

### `_normalize_provider`

Normalizes a provider identifier.

```python
_normalize_provider(
    provider: str
) -> str
```

It lowercases the text and replaces `-` with `_`.

## Examples

### Defer a Tool by Name

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ProviderToolSearchMiddleware

agent = create_agent(
    "anthropic:claude-opus-4-8",
    tools=[
        get_weather,
        send_email,
        lookup_order,
    ],
    middleware=[
        ProviderToolSearchMiddleware(
            searchable_tools=["lookup_order"]
        )
    ],
)
```

`lookup_order` is sent as a deferred tool, while the provider-native search tool is added to the request.

### Defer Multiple Tools

```python
middleware = ProviderToolSearchMiddleware(
    searchable_tools=[
        "lookup_order",
        "search_inventory",
        "find_customer",
    ]
)
```

All listed names must be present among the bound `BaseTool` instances.

### Pass Tool Instances

```python
middleware = ProviderToolSearchMiddleware(
    searchable_tools=[
        lookup_order,
        search_inventory,
    ]
)
```

The middleware stores their `.name` values internally.

### Mark a Tool Directly

```python
lookup_order.extras = {
    **(lookup_order.extras or {}),
    "defer_loading": True,
}

agent = create_agent(
    "openai:gpt-5.5",
    tools=[lookup_order],
    middleware=[
        ProviderToolSearchMiddleware()
    ],
)
```

A tool carrying the flag is deferred even when `searchable_tools` is empty.

### No Deferred Tools

```python
middleware = ProviderToolSearchMiddleware()
```

When none of the bound tools already has `defer_loading=True`, requests pass through unchanged.

## Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/provider_tool_search.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```